# 03 — NLP Analysis

**Mục tiêu:** Trích xuất thông tin quan trọng từ mô tả công việc bằng kỹ thuật NLP nhẹ và explainable.

**Input:** `data/cleaned/jobs_cleaned.csv`

---

## Nội dung
1. Skill Extraction (rule-based)
2. Keyword Extraction (TF-IDF)
3. Topic groups (lightweight keyword clustering)
4. Similarity & resume/job preparation


In [11]:
import sys
import pathlib
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))

from config import CLEANED_JOBS_FILE
from src.nlp import SkillExtractor, KeywordExtractor

# Load cleaned dataset for NLP analysis
# The dataset includes fields such as job_title, company_name, job_description,
# location, salary_midpoint_usd, and skill_count.

df = pd.read_csv(CLEANED_JOBS_FILE)
df = df.rename(columns=str.strip)
df = df[df['job_description'].notna()].reset_index(drop=True)

def infer_role_category(title: str) -> str:
    title_text = str(title or '').lower()
    if 'engineer' in title_text or 'developer' in title_text:
        return 'Engineering'
    if 'scientist' in title_text or 'machine learning' in title_text or 'ml' in title_text:
        return 'Data Science'
    if 'analyst' in title_text or 'analytics' in title_text:
        return 'Analytics'
    if 'product' in title_text:
        return 'Product'
    if 'manager' in title_text:
        return 'Management'
    if 'research' in title_text or 'researcher' in title_text:
        return 'Research'
    if 'consultant' in title_text:
        return 'Consulting'
    if 'business' in title_text:
        return 'Business'
    return 'Other'

df['role_category'] = df['job_title'].fillna('').apply(infer_role_category)

print('Loaded dataset:', df.shape)
print('Columns:', df.columns.tolist())
print('Non-empty job descriptions:', int(df['job_description'].notna().sum()))
print('Role categories sample:', df['role_category'].value_counts().head().to_dict())

df.head(3)


Loaded dataset: (6707, 37)
Columns: ['job_id', 'job_hash', 'job_title', 'company_name', 'company_name_normalized', 'salary', 'salary_min_usd', 'salary_max_usd', 'salary_midpoint_usd', 'salary_currency_norm', 'has_salary', 'is_negotiable', 'location', 'location_normalized', 'is_remote', 'employment_type', 'job_level', 'skills_str', 'skill_count', 'experience_years_parsed', 'experience_level_inferred', 'job_description', 'benefits', 'posted_date', 'posted_date_dt', 'expiry_date', 'expiry_date_dt', 'days_since_posted', 'url', 'source_website', 'industry', 'job_type', 'data_completeness', 'crawled_at', 'is_active', 'in_analysis_period', 'role_category']
Non-empty job descriptions: 6707
Role categories sample: {'Engineering': 3761, 'Other': 1684, 'Analytics': 643, 'Management': 226, 'Product': 186}


,job_id,job_hash,job_title,company_name,company_name_normalized,salary,salary_min_usd,salary_max_usd,salary_midpoint_usd,salary_currency_norm,...,days_since_posted,url,source_website,industry,job_type,data_completeness,crawled_at,is_active,in_analysis_period,role_category
0,528d6aa6a8ec26624b7d002083d39958,aeae9acaea2075edaff9ec3fb5b00c0b,Fullstack Developer,Cmc telecom,Cmc telecom,Đang cập nhật,NaN,NaN,NaN,NaN,...,1.0,https://123job.vn/viec-lam/fullstack-developer...,123job,IT / Technology,Full-time,full,2026-05-14T11:48:13.436280,NaN,NaN,Engineering
1,834b91cb6c26288a6503fd45d4b9a9f0,ff3edd77b9edcd1d48641477e7ec3564,Kỹ Sư Phân Tích Bảo Mật – Security Analyst/SOC...,Cmc telecom,Cmc telecom,Đang cập nhật,NaN,NaN,NaN,NaN,...,1.0,https://123job.vn/viec-lam/ky-su-phan-tich-bao...,123job,IT / Technology,Full-time,full,2026-05-14T11:51:43.441349,NaN,NaN,Analytics
2,ca22a5ce127012e864cb7976dbcb8578,4b950cde568b780ad1c700bf3aa6c449,Product Manager - Hệ thống Quản trị Nội bộ Sapo,Công ty cổ phần công nghệ sapo,công nghệ sapo,Đang cập nhật,NaN,NaN,NaN,NaN,...,2.0,https://123job.vn/viec-lam/product-manager-he-...,123job,IT / Technology,Full-time,full,2026-05-14T11:56:27.014955,NaN,NaN,Product


## 1. Skill Extraction

Sử dụng `SkillExtractor` rule-based để trích xuất kỹ năng chính trong mô tả công việc. Kết quả được chuẩn hóa theo taxonomy kỹ năng và alias để đảm bảo tính nhất quán.


In [12]:
extractor = SkillExtractor()

df['extracted_skills'] = extractor.extract_batch(df, col='job_description')

skill_freq = extractor.frequency(df['job_description'].fillna('').tolist(), top_n=25)
print('Top 25 extracted skills:')
skill_freq

print('\nSample extracted skills for the first jobs:')
df.loc[:, ['job_title', 'company_name', 'role_category', 'extracted_skills']].head(8)


Top 25 extracted skills:

Sample extracted skills for the first jobs:


,job_title,company_name,role_category,extracted_skills
0,Fullstack Developer,Cmc telecom,Engineering,"[Agile, Angular, AWS, GCP, Java, JavaScript, M..."
1,Kỹ Sư Phân Tích Bảo Mật – Security Analyst/SOC...,Cmc telecom,Analytics,[R]
2,Product Manager - Hệ thống Quản trị Nội bộ Sapo,Công ty cổ phần công nghệ sapo,Product,[R]
3,Lập trình viên Android,Công ty cổ phần giải pháp thanh toán việt nam ...,Other,[Java]
4,Senior/Leader AI Engineer - Đà Nẵng,Vnext software,Engineering,[]
5,Backend Developer (Nodejs/Nestjs) – Từ 6 Tháng...,Công ty cổ phần đầu tư thương mại và dịch vụ b...,Engineering,"[Docker, MySQL, Node.js, Redis]"
6,Senior AI Engineer,Công ty cổ phần vàng bạc đá quý phú nhuận,Engineering,"[CI/CD, LangChain, R, React]"
7,AI Product Manager,Công ty cổ phần vàng bạc đá quý phú nhuận,Product,[]


## 2. Keyword Extraction (TF-IDF)

Sử dụng `KeywordExtractor` để trích xuất các token quan trọng từ mô tả công việc. Bộ công cụ dùng TF-IDF với fallback tần suất để giữ lightweight và explainable.


In [13]:
keyword_extractor = KeywordExtractor(max_features=350, ngram_range=(1, 2), min_df=3)
keyword_extractor.fit(df['job_description'].fillna('').tolist())

top_keywords = keyword_extractor.top_keywords(n=30)
print('Top 30 keywords from job descriptions:')
top_keywords

sample_doc = df.loc[0, 'job_description']
sample_keywords = keyword_extractor.keywords_for_document(sample_doc, n=15)
print('\nTop keywords for the first job description:')
print(sample_keywords)


c:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.26.3)
  from scipy.sparse import csr_matrix, issparse


Top 30 keywords from job descriptions:

Top keywords for the first job description:
['lương', 'nghệ', 'năm', 'kỹ', 'năng', 'ngày', 'thưởng', 'kỹ năng', 'cơ', 'trúc', 'như', 'nhật', 'kiến', 'ngành', 'kiến trúc']


## 3. Topic groups (lightweight keyword clustering)
Thay vì dùng BERTopic hay embedding nặng, notebook tạo nhóm chủ đề dựa trên top keywords đã trích xuất.


In [14]:
topic_groups = keyword_extractor.topic_model(df['job_description'].fillna('').tolist(), n_topics=6)
print('Lightweight topic groups derived from keywords:')
topic_groups

role_keywords = keyword_extractor.role_keyword_associations(
    df,
    text_col='job_description',
    role_col='role_category',
    top_n=6,
)
print('\nTop keywords by role category:')
role_keywords.head(20)


Lightweight topic groups derived from keywords:

Top keywords by role category:


,role,keyword,score
0,Analytics,data,0.107334
1,Analytics,liệu,0.084688
2,Analytics,business,0.077011
3,Analytics,dữ,0.074183
4,Analytics,dữ liệu,0.074149
5,Analytics,phân,0.061218
6,Business,liệu,0.119708
7,Business,dữ,0.107225
8,Business,dữ liệu,0.106323
9,Business,data,0.084920


## 4. Similarity & Resume/Job Preparation
Sử dụng những kỹ năng và keyword đã trích để đánh giá độ gần nhau của các tài liệu. Đây là nền tảng cho resume/JD matching nhẹ, explainable.


In [15]:
if len(df) > 1:
    sample_skills = set(df.loc[0, 'extracted_skills'])
    compare_skills = set(df.loc[1, 'extracted_skills'])

    jaccard_similarity = (
        len(sample_skills & compare_skills) / len(sample_skills | compare_skills)
        if sample_skills or compare_skills
        else 0
    )
    print('Sample job 1 skills:', df.loc[0, 'extracted_skills'])
    print('Sample job 2 skills:', df.loc[1, 'extracted_skills'])
    print(f'Jaccard similarity between job 1 and job 2 skills: {jaccard_similarity:.2f}')

    sample_doc = df.loc[0, 'job_description']
    compare_doc = df.loc[1, 'job_description']
    sample_keywords_set = set(keyword_extractor.keywords_for_document(sample_doc, n=20))
    compare_keywords_set = set(keyword_extractor.keywords_for_document(compare_doc, n=20))

    keyword_overlap = sample_keywords_set & compare_keywords_set
    print('\nKeyword overlap between job 1 and job 2:')
    print(keyword_overlap)
else:
    print('Not enough rows to compare similarity.')


Sample job 1 skills: ['Agile', 'Angular', 'AWS', 'GCP', 'Java', 'JavaScript', 'MongoDB', 'MySQL', 'React', 'Scrum']
Sample job 2 skills: ['R']
Jaccard similarity between job 1 and job 2 skills: 0.00

Keyword overlap between job 1 and job 2:
{'cơ', 'kiến'}


## Kết luận

- `SkillExtractor` cho phép trích xuất kỹ năng explainable từ mô tả công việc.
- `KeywordExtractor` hỗ trợ trích keyword TF-IDF và tạo nhóm chủ đề nhẹ.
- Notebook giữ workflow nhẹ, phù hợp với Phase 6: analytics-oriented và không lệ thuộc transformer.
- Tiếp theo, có thể mở rộng phần này sang resume/JD matching bằng cách so sánh skill/keyword overlaps và thêm scoring.
